# FlexSDS Server trên Google Colab (vLLM Speculative Decoding Engine)

Notebook này hỗ trợ khởi chạy máy chủ phục vụ FlexSDS với kiến trúc giải mã đầu cơ thích ứng EAGLE-3.1 trên Google Colab (GPU T4 / L4 / A100) và mở đường hầm Cloudflare Tunnel để kết nối với `flexsds CLI`.

In [ ]:
# BƯỚC 1: ĐỒNG BỘ MÃ NGUỒN VÀ CÀI ĐẶT MÔI TRƯỜNG
import os
import subprocess

REPO_URL = "https://github.com/theflyingkhui04/do-an-ptit-d22.git"
TARGET_DIR = "/content/do-an-ptit-d22"

if not os.path.exists(TARGET_DIR):
    print("==> Đang tải mã nguồn repo FlexSDS...")
    subprocess.run(["git", "clone", REPO_URL, TARGET_DIR], check=True)
else:
    print("==> Đồng bộ mã nguồn mới nhất...")
    subprocess.run(["git", "-C", TARGET_DIR, "pull"], check=True)

print("==> Đang cài đặt vLLM v0.30.0 và các thư viện cần thiết...")
subprocess.run(["pip", "install", "-q", "-r", f"{TARGET_DIR}/requirements-server.txt"], check=True)
subprocess.run(["pip", "uninstall", "-y", "torchaudio"], check=False)
print("✔ Hoàn tất cài đặt môi trường!")

In [ ]:
# BƯỚC 2: CÀI ĐẶT CLOUDFLARE TUNNEL (CLOUDFLARED)
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null
print("✔ Cloudflared đã sẵn sàng!")

In [ ]:
# BƯỚC 3: KHỞI CHẠY vLLM SERVING VÀ ĐƯỜNG HẦM CLOUDFLARE
import os
import time
import subprocess

os.environ["PYTHONPATH"] = f"{TARGET_DIR}:{os.environ.get('PYTHONPATH', '')}"
os.environ["ADAPTIVE_ENABLE_VRAM_GUARD"] = "1"

server_cmd = [
    "vllm", "serve", "Qwen/Qwen2.5-7B-Instruct-AWQ",
    "--host", "0.0.0.0",
    "--port", "8000",
    "--served-model-name", "qwen2.5-7b-adaptive",
    "--quantization", "awq",
    "--dtype", "half",
    "--gpu-memory-utilization", "0.85",
    "--max-model-len", "4096",
    "--speculative-config", '{"method": "eagle3", "model": "duckhoi/qwen2.5-7b-eagle3.1-speculator", "num_speculative_tokens": 4, "use_heterogeneous_vocab": true, "draft_sample_method": "greedy"}',
    "--scheduler-cls", "src.server.adaptive_scheduler.AdaptiveAsyncScheduler",
    "--distributed-executor-backend", "src.server.lineage_executor.LineageUniProcExecutor",
    "--per-request-spec-decode-metrics", "summary",
]

print("==> Đang khởi động máy chủ vLLM chạy nền...")
vllm_proc = subprocess.Popen(server_cmd)

print("==> Đang kích hoạt Cloudflare Tunnel công khai...")
tunnel_cmd = ["cloudflared", "tunnel", "--url", "http://localhost:8000"]
subprocess.run(tunnel_cmd)